# Tutor Assistant - LangChain Walkthrough Notebook

This notebook walks you through the core capabilities from `requirements.md` and gives you an end-to-end test setup for handwriting recognition from your PDF notes using Bedrock + Claude 3.5 Sonnet.

Target stack:
- LangChain orchestration
- Amazon Bedrock (`anthropic.claude-3-5-sonnet-20240620-v1:0`)
- `pdf2image` for PDF -> image conversion
- Local test mode first, then extension to Google Drive/Calendar


## 1) Environment setup (UV)

Run these commands in your terminal (outside notebook):

```bash
uv venv
source .venv/bin/activate
uv pip install -U jupyter ipykernel boto3 pdf2image pillow pydantic python-dotenv
uv pip install -U langchain langchain-core langchain-aws langchain-community
# optional for benchmark extension
uv pip install -U python-Levenshtein pandas matplotlib
```

macOS dependency for pdf2image:
```bash
brew install poppler
```


## 2) Expected env vars

Create `.env` file (or export vars in shell):

```dotenv
AWS_REGION=us-east-1
AWS_ACCESS_KEY_ID=...
AWS_SECRET_ACCESS_KEY=...
# optional
AWS_SESSION_TOKEN=...
```

If you use AWS SSO/profile, keep credentials in AWS config and skip direct keys.


In [ ]:
import os
import io
import re
import json
import time
import base64
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pdf2image import convert_from_path
from PIL import Image

import boto3
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"

print("AWS_REGION:", AWS_REGION)
print("MODEL_ID:", MODEL_ID)


## 3) Output schema for math note analysis

This schema maps to FR1 and helps normalize output for FR2 progress tracking.


In [ ]:
class NoteAnalysis(BaseModel):
    topics: list[str] = Field(default_factory=list, description="Detected math topics")
    tasks: list[str] = Field(default_factory=list, description="Detected exercises/problems")
    errors: list[str] = Field(default_factory=list, description="Potential solution mistakes")
    summary: str = Field(default="", description="Short summary of the page")

parser = JsonOutputParser(pydantic_object=NoteAnalysis)
format_instructions = parser.get_format_instructions()
format_instructions[:400]


## 4) PDF -> images helpers

Use your own PDF path in the next cell.


In [ ]:
def pdf_to_images(pdf_path: str | Path, dpi: int = 220) -> list[Image.Image]:
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"File not found: {pdf_path}")
    return convert_from_path(str(pdf_path), dpi=dpi)

def image_to_base64_png(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


## 5) Bedrock call (direct boto3) - best for debugging

This is your quick MVP test for handwriting recognition from PDF notes.


In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

def safe_json_extract(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)

def analyze_page_with_claude(image: Image.Image, max_tokens: int = 1800) -> dict[str, Any]:
    img_b64 = image_to_base64_png(image)

    prompt = (
        "You are an expert math tutor assistant. Analyze this handwritten math note page. \
Return strictly valid JSON with keys: topics, tasks, errors, summary. \
topics/tasks/errors must be arrays of strings; summary must be a string."
    )

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": img_b64
                        }
                    },
                    {"type": "text", "text": prompt}
                ]
            }
        ]
    }

    response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(body))
    payload = json.loads(response["body"].read())
    text = payload["content"][0]["text"]
    return safe_json_extract(text)


In [ ]:
# Change this path to your own notes PDF
PDF_PATH = "data/sample_notes.pdf"

images = pdf_to_images(PDF_PATH, dpi=220)
print(f"Loaded pages: {len(images)}")

# preview first page
images[0]


In [ ]:
t0 = time.time()
first_page_result = analyze_page_with_claude(images[0])
latency_s = time.time() - t0

validated = NoteAnalysis.model_validate(first_page_result)
print(f"Latency: {latency_s:.2f}s")
validated.model_dump()


## 6) LangChain wrapper (same model, cleaner orchestration)

This part is useful when you start connecting FR2/FR3/FR4/FR5 in one workflow.


In [ ]:
llm = ChatBedrockConverse(
    model=MODEL_ID,
    region_name=AWS_REGION,
    max_tokens=1800,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert math tutor assistant. Return only valid JSON."),
    ("human", "Analyze this OCR text from handwritten notes: {ocr_text}\n{format_instructions}")
])

chain = prompt | llm | parser

# Example: when you already have OCR text from an external OCR engine
example_ocr = "x^2 - 5x + 6 = 0; student solved x=1 and x=6; one root seems wrong"
structured = chain.invoke({"ocr_text": example_ocr, "format_instructions": format_instructions})
structured


## 7) Minimal FR1-FR5 function scaffold

These are local notebook stubs that map directly to your required architecture and can be replaced with real Google/Twilio API calls later.


In [ ]:
def fetch_latest_notes(student_id: str, local_pdf_path: str) -> list[Image.Image]:
    # TODO: replace with Google Drive API lookup by student folder
    print(f"[fetch_latest_notes] student_id={student_id}, source={local_pdf_path}")
    return pdf_to_images(local_pdf_path, dpi=220)

def analyze_math_content(images: list[Image.Image], max_pages: int | None = 3) -> list[NoteAnalysis]:
    results: list[NoteAnalysis] = []
    subset = images if max_pages is None else images[:max_pages]
    for idx, img in enumerate(subset, start=1):
        raw = analyze_page_with_claude(img)
        parsed = NoteAnalysis.model_validate(raw)
        print(f"[analyze_math_content] page={idx} topics={len(parsed.topics)} errors={len(parsed.errors)}")
        results.append(parsed)
    return results

def get_daily_schedule() -> list[dict[str, str]]:
    # TODO: replace with Google Calendar API
    return [
        {"time": "09:00", "student": "Student_A", "topic_hint": "Quadratic equations"},
        {"time": "17:00", "student": "Student_B", "topic_hint": "Trigonometry"},
    ]

def update_student_progress_file(student_id: str, analyses: list[NoteAnalysis], out_dir: str = "outputs") -> Path:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    merged_topics = sorted({t for a in analyses for t in a.topics})
    merged_errors = [e for a in analyses for e in a.errors]

    payload = {
        "student_id": student_id,
        "topics_seen": merged_topics,
        "critical_gaps": merged_errors,
        "pages_analyzed": len(analyses),
        "updated_at_epoch": int(time.time()),
    }

    dst = out / f"progress_{student_id}.json"
    dst.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"[update_student_progress_file] wrote: {dst}")
    return dst


In [ ]:
# End-to-end local simulation for FR1 + FR2
student_id = "student_demo"
images = fetch_latest_notes(student_id, PDF_PATH)
analyses = analyze_math_content(images, max_pages=2)
progress_file = update_student_progress_file(student_id, analyses)
progress_file


## 8) Quick benchmark harness (quality + latency)

Use this to compare candidate approaches later (Claude direct vs Google Document AI vs Mathpix).


In [ ]:
def benchmark_claude_on_pdf(pdf_path: str, pages: int = 3) -> list[dict[str, Any]]:
    imgs = pdf_to_images(pdf_path, dpi=220)[:pages]
    rows: list[dict[str, Any]] = []

    for i, img in enumerate(imgs, start=1):
        t0 = time.time()
        result = analyze_page_with_claude(img)
        dt = time.time() - t0

        parsed = NoteAnalysis.model_validate(result)
        rows.append({
            "page": i,
            "latency_s": round(dt, 2),
            "topics_n": len(parsed.topics),
            "tasks_n": len(parsed.tasks),
            "errors_n": len(parsed.errors),
            "summary": parsed.summary[:160],
        })
    return rows

bench = benchmark_claude_on_pdf(PDF_PATH, pages=2)
bench


## 9) Model recommendation for your handwriting-notes MVP

Recommended default: **Bedrock Claude 3.5 Sonnet** for first production iteration.

Why for your case:
- Native fit with your required stack (AWS + LangChain).
- Single multimodal step (OCR + reasoning + error detection) gives faster MVP.
- Easy to evolve into FR2-FR5 orchestration with structured outputs.

Strong fallback for OCR fidelity benchmark:
- Google Document AI (Math)
- Mathpix

Use the benchmark cell pattern above to compare latency/cost/quality on 5-10 pages of your own notes.
